In [1]:
import os
os.chdir('/home/smallyan/eval_agent')
print(f"Working directory: {os.getcwd()}")

Working directory: /home/smallyan/eval_agent


# Replicator-Documentation Evaluator

This notebook evaluates whether the replicator's documentation faithfully reproduces the results and conclusions of the original experiment.

## Task Overview
1. Compare original documentation with replicated documentation
2. Evaluate result fidelity (DE1)
3. Evaluate conclusion consistency (DE2)
4. Check for external/hallucinated information (DE3)
5. Generate evaluation summary files

In [2]:
# Check GPU availability
import torch
if torch.cuda.is_available():
    print(f"GPU available: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
else:
    print("No GPU available, using CPU")

GPU available: NVIDIA A100 80GB PCIe
CUDA version: 12.8


In [3]:
# Define paths
original_repo = "/net/scratch2/smallyan/belief-tracking_eval"
replication_dir = os.path.join(original_repo, "evaluation", "replications")

# Check if directories exist
print(f"Original repo exists: {os.path.exists(original_repo)}")
print(f"Replication dir exists: {os.path.exists(replication_dir)}")

# List contents
if os.path.exists(original_repo):
    print(f"\nOriginal repo contents:")
    for item in os.listdir(original_repo):
        print(f"  {item}")

if os.path.exists(replication_dir):
    print(f"\nReplication dir contents:")
    for item in os.listdir(replication_dir):
        print(f"  {item}")

Original repo exists: False
Replication dir exists: False


In [4]:
# Let's search for belief-tracking related directories
import subprocess

# Search for belief-tracking directories
result = subprocess.run(['find', '/net', '-name', '*belief-tracking*', '-type', 'd', '-maxdepth', 4], 
                       capture_output=True, text=True, timeout=30)
print("Directories with 'belief-tracking':")
print(result.stdout if result.stdout else "None found")
print(result.stderr if result.stderr else "")

TypeError: expected str, bytes or os.PathLike object, not int

In [5]:
# Fix the find command - maxdepth needs to come before other predicates
result = subprocess.run(['find', '/net', '-maxdepth', '4', '-name', '*belief-tracking*', '-type', 'd'], 
                       capture_output=True, text=True, timeout=60)
print("Directories with 'belief-tracking':")
print(result.stdout if result.stdout else "None found")
if result.stderr:
    print("Errors:", result.stderr[:500])

Directories with 'belief-tracking':
/net/scratch2/smallyan/belief_tracking_eval/belief-tracking_claude_2025-12-23_22-28-22

Errors: find: ‘/net/monsoon/qiang’: Permission denied
find: ‘/net/monsoon/S2S’: Permission denied
find: ‘/net/monsoon/.Trash-27283’: Permission denied
find: ‘/net/monsoon/Diana’: Permission denied
find: ‘/net/monsoon/bing’: Permission denied
find: ‘/net/monsoon/marchakitus’: Permission denied
find: ‘/net/monsoon/PlaSim’: Permission denied
find: ‘/net/monsoon/weidong’: Permission denied
find: ‘/net/monsoon/jesswan/AI4SRM’: Permission denied
find: ‘/net/monsoon/.vscode’: Permission denied
find: ‘/net/mons


In [6]:
# Found the correct path - it's belief_tracking_eval (with underscore)
# Let's explore the structure
belief_tracking_base = "/net/scratch2/smallyan/belief_tracking_eval"
print(f"Base directory exists: {os.path.exists(belief_tracking_base)}")

if os.path.exists(belief_tracking_base):
    print(f"\nContents of {belief_tracking_base}:")
    for item in os.listdir(belief_tracking_base):
        full_path = os.path.join(belief_tracking_base, item)
        if os.path.isdir(full_path):
            print(f"  [DIR] {item}")
        else:
            print(f"  [FILE] {item}")

Base directory exists: True

Contents of /net/scratch2/smallyan/belief_tracking_eval:
  [FILE] plan.md
  [DIR] data
  [DIR] evaluation
  [DIR] src
  [DIR] belief-tracking_claude_2025-12-23_22-28-22
  [FILE] pyproject.toml
  [DIR] scripts
  [FILE] uv.lock
  [FILE] causalmodel_novis.png
  [DIR] results
  [DIR] notebooks
  [FILE] .python-version
  [DIR] .git
  [FILE] documentation.pdf
  [FILE] CodeWalkthrough.md
  [FILE] .gitignore


In [7]:
# Check for documentation.md or equivalent in root
original_repo = "/net/scratch2/smallyan/belief_tracking_eval"

# Look for documentation files
doc_files = []
for item in os.listdir(original_repo):
    if 'doc' in item.lower() or 'readme' in item.lower() or item.endswith('.md'):
        doc_files.append(item)
        
print("Documentation files in root:")
for f in doc_files:
    print(f"  {f}")

# Check the evaluation directory
eval_dir = os.path.join(original_repo, "evaluation")
print(f"\nContents of evaluation directory:")
if os.path.exists(eval_dir):
    for item in os.listdir(eval_dir):
        full_path = os.path.join(eval_dir, item)
        if os.path.isdir(full_path):
            print(f"  [DIR] {item}")
        else:
            print(f"  [FILE] {item}")

Documentation files in root:
  plan.md
  documentation.pdf
  CodeWalkthrough.md

Contents of evaluation directory:
  [FILE] generalization_eval.ipynb
  [FILE] code_critic_evaluation.ipynb
  [FILE] self_matching.ipynb
  [FILE] code_critic_summary.json
  [FILE] consistency_evaluation.json
  [FILE] generalization_eval_summary.json
  [DIR] replications


In [8]:
# Check the replications directory
replications_dir = os.path.join(original_repo, "evaluation", "replications")
print(f"Contents of replications directory:")
if os.path.exists(replications_dir):
    for item in os.listdir(replications_dir):
        full_path = os.path.join(replications_dir, item)
        if os.path.isdir(full_path):
            print(f"  [DIR] {item}")
        else:
            print(f"  [FILE] {item}")

Contents of replications directory:
  [FILE] evaluation_replication.md
  [FILE] documentation_replication.md
  [FILE] self_replication_evaluation.json
  [FILE] experiment_results.json
  [FILE] iia_results.png
  [FILE] replication.ipynb


In [9]:
# Great! Found documentation_replication.md in replications
# Now we need to find the original documentation
# Let's check if there's a documentation.md or use CodeWalkthrough.md

# First, let's check what documentation.pdf contains by looking for text files
# and check if there's a markdown version

# Let's also check the belief-tracking subdirectory
bt_subdir = os.path.join(original_repo, "belief-tracking_claude_2025-12-23_22-28-22")
print(f"Contents of belief-tracking subdirectory:")
if os.path.exists(bt_subdir):
    for item in os.listdir(bt_subdir):
        print(f"  {item}")
        
# Check notebooks directory
notebooks_dir = os.path.join(original_repo, "notebooks")
print(f"\nContents of notebooks directory:")
if os.path.exists(notebooks_dir):
    for item in os.listdir(notebooks_dir):
        print(f"  {item}")

Contents of belief-tracking subdirectory:
  results
  logs
  notebooks

Contents of notebooks directory:
  causal_subspace_analysis
  attn_knockout
  causalToM_vis
  bigToM
  causalToM_novis


In [10]:
# Let's look for documentation.md in the original repo and subdirectories
import glob

# Search for documentation.md
doc_md_files = glob.glob(os.path.join(original_repo, "**", "documentation.md"), recursive=True)
print("Files named documentation.md:")
for f in doc_md_files:
    print(f"  {f}")

# Also search for any .md files
all_md_files = glob.glob(os.path.join(original_repo, "**", "*.md"), recursive=True)
print(f"\nAll markdown files:")
for f in all_md_files:
    print(f"  {f}")

Files named documentation.md:

All markdown files:
  /net/scratch2/smallyan/belief_tracking_eval/plan.md
  /net/scratch2/smallyan/belief_tracking_eval/CodeWalkthrough.md
  /net/scratch2/smallyan/belief_tracking_eval/evaluation/replications/evaluation_replication.md
  /net/scratch2/smallyan/belief_tracking_eval/evaluation/replications/documentation_replication.md


In [11]:
# The original documentation seems to be in PDF format (documentation.pdf)
# Let's also check CodeWalkthrough.md and plan.md as potential original documentation

# First, read CodeWalkthrough.md
codewalkthrough_path = os.path.join(original_repo, "CodeWalkthrough.md")
with open(codewalkthrough_path, 'r') as f:
    codewalkthrough_content = f.read()
    
print("CodeWalkthrough.md content:")
print("="*80)
print(codewalkthrough_content[:3000])
print("...")
print(f"\n[Total length: {len(codewalkthrough_content)} characters]")

CodeWalkthrough.md content:
# Language Models use Lookbacks to Track Beliefs

This repository contains the code and experiments for the paper ["Language Models use Lookbacks to Track Beliefs"](https://arxiv.org/abs/2505.14685) by Prakash et al, 2025. The work investigates how language models (specifically Llama-3-70B-Instruct and Llama-3.1-405B-Instruct) represent and track characters' beliefs.


![Causal Model in No-Visibility Setting](causalmodel_novis.png)


Please check [belief.baulab.info](https://belief.baulab.info/) for more information.

## Setup

1. Clone the repository:
```bash
git clone https://github.com/Nix07/belief_tracking.git
cd belief_tracking
```

2. Set up the environment:
```bash
uv sync
source .venv/bin/activate
```

3. Configure `env.yml` with following environment variables:
- Set `NDIF_KEY` for API access
- Set `HF_WRITE` for Hugging Face access

4. To perform subspace level analysis, you would need singular vectors that you can request by sending an email to [N

In [12]:
# Read the full CodeWalkthrough.md
print(codewalkthrough_content)

# Language Models use Lookbacks to Track Beliefs

This repository contains the code and experiments for the paper ["Language Models use Lookbacks to Track Beliefs"](https://arxiv.org/abs/2505.14685) by Prakash et al, 2025. The work investigates how language models (specifically Llama-3-70B-Instruct and Llama-3.1-405B-Instruct) represent and track characters' beliefs.


![Causal Model in No-Visibility Setting](causalmodel_novis.png)


Please check [belief.baulab.info](https://belief.baulab.info/) for more information.

## Setup

1. Clone the repository:
```bash
git clone https://github.com/Nix07/belief_tracking.git
cd belief_tracking
```

2. Set up the environment:
```bash
uv sync
source .venv/bin/activate
```

3. Configure `env.yml` with following environment variables:
- Set `NDIF_KEY` for API access
- Set `HF_WRITE` for Hugging Face access

4. To perform subspace level analysis, you would need singular vectors that you can request by sending an email to [Nikhil](https://nix07.github.

In [13]:
# Now let's read the documentation_replication.md from replications
doc_replication_path = os.path.join(original_repo, "evaluation", "replications", "documentation_replication.md")
with open(doc_replication_path, 'r') as f:
    doc_replication_content = f.read()
    
print("documentation_replication.md content:")
print("="*80)
print(doc_replication_content)

documentation_replication.md content:
# Belief Tracking Replication Documentation

## Goal

Replicate the key findings from "Language Models use Lookbacks to Track Beliefs" (Prakash et al., 2025), specifically the causal intervention experiments that localize:
1. **Answer Pointer**: The mechanism that directs the model to retrieve the correct state token
2. **Answer Payload**: The actual state value that the model outputs

## Data

### Dataset: CausalToM
- **Source**: Synthetic belief tracking scenarios in a restaurant setting
- **Structure**: Stories about two characters filling containers with different drinks
- **Templates**: Template 2 used (no explicit visibility constraints in story text)
- **Entities**:
  - 103 character names
  - 21 container types (cup, mug, bottle, etc.)
  - 23 drink types (water, coffee, wine, etc.)

### Counterfactual Generation
Two types of counterfactual pairs were generated:

1. **Pointer Counterfactuals**:
   - Reverses sentence order and uses different

In [14]:
# Let's also check plan.md as it may have more details about the original experiment
plan_path = os.path.join(original_repo, "plan.md")
with open(plan_path, 'r') as f:
    plan_content = f.read()
    
print("plan.md content:")
print("="*80)
print(plan_content)

plan.md content:
# Plan
## Objective
Analyze how language models internally represent and track beliefs of characters, particularly when those beliefs may differ from reality, using causal mediation and abstraction methods to uncover systematic computational mechanisms.

## Hypothesis
1. Language models use a lookback mechanism to track beliefs, where reference information is copied to two locations (address and pointer) enabling later retrieval of important information when needed.
2. The model assigns ordering IDs to character, object, and state tokens, then binds character-object-state triples together by co-locating their reference information in low-rank subspaces.
3. A binding lookback retrieves the correct state OI using character and object OIs, and an answer lookback retrieves the corresponding state token.
4. When visibility information is provided, a visibility lookback uses a visibility ID to retrieve information about the observed character and update the observing charact

In [15]:
# Now let's check the experiment_results.json to see the actual replicated results
import json

experiment_results_path = os.path.join(original_repo, "evaluation", "replications", "experiment_results.json")
with open(experiment_results_path, 'r') as f:
    experiment_results = json.load(f)
    
print("experiment_results.json:")
print(json.dumps(experiment_results, indent=2))

experiment_results.json:
{
  "model": "meta-llama/Llama-3.1-8B-Instruct",
  "num_layers": 32,
  "seed": 42,
  "n_samples": 20,
  "pointer_experiment": {
    "layers_tested": [
      0,
      4,
      8,
      12,
      16,
      20,
      24,
      28
    ],
    "results": {
      "0": 0.05,
      "4": 0.05,
      "8": 0.05,
      "12": 0.35,
      "16": 0.8,
      "20": 0.7,
      "24": 0.05,
      "28": 0.0
    },
    "peak_layer": 16,
    "peak_iia": 0.8
  },
  "payload_experiment": {
    "layers_tested": [
      0,
      2,
      4,
      6,
      8,
      10,
      12,
      14,
      16,
      18,
      20,
      22,
      24,
      26,
      28,
      30
    ],
    "results": {
      "0": 0.0,
      "2": 0.0,
      "4": 0.0,
      "6": 0.0,
      "8": 0.0,
      "10": 0.0,
      "12": 0.0,
      "14": 0.0,
      "16": 0.0,
      "18": 0.0,
      "20": 0.0,
      "22": 0.0,
      "24": 0.3,
      "26": 0.7,
      "28": 0.9,
      "30": 0.9
    },
    "peak_layer": 28,
    "peak_i

In [16]:
# Let's also check for any original results in the results directory
results_dir = os.path.join(original_repo, "results")
print(f"Contents of results directory:")
if os.path.exists(results_dir):
    for item in os.listdir(results_dir):
        full_path = os.path.join(results_dir, item)
        if os.path.isdir(full_path):
            print(f"  [DIR] {item}")
        else:
            print(f"  [FILE] {item}")

Contents of results directory:
  [DIR] causalToM_novis
  [DIR] bigToM
  [DIR] causal_mediation_analysis
  [DIR] attn_knockout
  [DIR] model_evaluations
  [DIR] causalToM_vis


In [17]:
# Check causalToM_novis results
causalToM_novis_dir = os.path.join(results_dir, "causalToM_novis")
print(f"Contents of causalToM_novis:")
if os.path.exists(causalToM_novis_dir):
    for item in os.listdir(causalToM_novis_dir):
        full_path = os.path.join(causalToM_novis_dir, item)
        if os.path.isdir(full_path):
            print(f"  [DIR] {item}")
        else:
            print(f"  [FILE] {item}")

Contents of causalToM_novis:
  [DIR] Meta-Llama-3.1-405B-Instruct-8bit
  [DIR] Meta-Llama-3-70B-Instruct
  [DIR] Qwen2.5-14B-Instruct


In [18]:
# Check the 70B results which matches the original paper
llama70b_dir = os.path.join(causalToM_novis_dir, "Meta-Llama-3-70B-Instruct")
print(f"Contents of Meta-Llama-3-70B-Instruct:")
if os.path.exists(llama70b_dir):
    for item in os.listdir(llama70b_dir):
        full_path = os.path.join(llama70b_dir, item)
        if os.path.isdir(full_path):
            print(f"  [DIR] {item}")
        else:
            print(f"  [FILE] {item}")

Contents of Meta-Llama-3-70B-Instruct:
  [DIR] answer_lookback
  [DIR] binding_lookback


In [19]:
# Check answer_lookback results
answer_lookback_dir = os.path.join(llama70b_dir, "answer_lookback")
print(f"Contents of answer_lookback:")
if os.path.exists(answer_lookback_dir):
    for item in os.listdir(answer_lookback_dir):
        print(f"  {item}")
        
# Try to read JSON results if any
json_files = [f for f in os.listdir(answer_lookback_dir) if f.endswith('.json')]
print(f"\nJSON files: {json_files}")

Contents of answer_lookback:
  pointer
  payload

JSON files: []


In [20]:
# Check pointer and payload subdirectories
pointer_dir = os.path.join(answer_lookback_dir, "pointer")
payload_dir = os.path.join(answer_lookback_dir, "payload")

print(f"Contents of pointer directory:")
for item in os.listdir(pointer_dir):
    print(f"  {item}")
    
print(f"\nContents of payload directory:")
for item in os.listdir(payload_dir):
    print(f"  {item}")

Contents of pointer directory:
  46.json
  20.json
  42.json
  38.json
  34.json
  56.json
  30.json
  52.json
  35.json
  57.json
  31.json
  53.json
  47.json
  10.json
  43.json
  39.json
  37.json
  55.json
  49.json
  33.json
  51.json
  60.json
  45.json
  59.json
  41.json
  70.json
  44.json
  58.json
  40.json
  0.json
  36.json
  54.json
  48.json
  32.json
  79.json
  50.json

Contents of payload directory:
  72.json
  10.json
  76.json
  62.json
  53.json
  57.json
  66.json
  63.json
  52.json
  30.json
  56.json
  67.json
  73.json
  20.json
  77.json
  61.json
  50.json
  79.json
  54.json
  65.json
  71.json
  40.json
  69.json
  58.json
  75.json
  0.json
  70.json
  68.json
  59.json
  74.json
  60.json
  51.json
  78.json
  55.json
  64.json


In [21]:
# Read some pointer results to understand the format
import json

# Read a few pointer files
pointer_results = {}
for layer_file in sorted(os.listdir(pointer_dir), key=lambda x: int(x.replace('.json', ''))):
    layer = int(layer_file.replace('.json', ''))
    with open(os.path.join(pointer_dir, layer_file), 'r') as f:
        data = json.load(f)
        pointer_results[layer] = data

# Show structure of one result
print("Sample pointer result (layer 40):")
print(json.dumps(pointer_results[40], indent=2)[:1500])

Sample pointer result (layer 40):
{
  "full_rank": {
    "accuracy": 0.9875,
    "rank": null,
    "metadata": null
  },
  "singular_vector": {
    "accuracy": 0.95,
    "rank": 12.0,
    "metadata": {
      "training_args": {
        "learning_rate": 0.1,
        "n_epochs": 1,
        "lamb": 0.1
      },
      "mask": [
        0.0,
        0.0,
        1.0,
        0.0,
        1.0,
        1.0,
        1.0,
        1.0,
        1.0,
        1.0,
        1.0,
        0.0,
        1.0,
        1.0,
        1.0,
        0.0,
        1.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,

In [22]:
# Extract and summarize the original 70B pointer results (full_rank accuracy = IIA)
print("Original 70B Pointer Results (by layer):")
print("-" * 40)
layers_sorted = sorted(pointer_results.keys())
for layer in layers_sorted:
    iia = pointer_results[layer]['full_rank']['accuracy']
    print(f"Layer {layer:2d}: IIA = {iia:.4f}")

# Find peak
peak_layer = max(pointer_results.keys(), key=lambda l: pointer_results[l]['full_rank']['accuracy'])
peak_iia = pointer_results[peak_layer]['full_rank']['accuracy']
print(f"\nPeak: Layer {peak_layer} with IIA = {peak_iia}")

Original 70B Pointer Results (by layer):
----------------------------------------
Layer  0: IIA = 0.0000
Layer 10: IIA = 0.0000
Layer 20: IIA = 0.0000
Layer 30: IIA = 0.0000
Layer 31: IIA = 0.0125
Layer 32: IIA = 0.0875
Layer 33: IIA = 0.5375
Layer 34: IIA = 0.9250
Layer 35: IIA = 0.8375
Layer 36: IIA = 0.8625
Layer 37: IIA = 0.9250
Layer 38: IIA = 1.0000
Layer 39: IIA = 0.9875
Layer 40: IIA = 0.9875
Layer 41: IIA = 0.9750
Layer 42: IIA = 0.9875
Layer 43: IIA = 0.9750
Layer 44: IIA = 0.9750
Layer 45: IIA = 0.9750
Layer 46: IIA = 0.9750
Layer 47: IIA = 0.9625
Layer 48: IIA = 0.9625
Layer 49: IIA = 0.9875
Layer 50: IIA = 0.9750
Layer 51: IIA = 0.9750
Layer 52: IIA = 0.9250
Layer 53: IIA = 0.7250
Layer 54: IIA = 0.5250
Layer 55: IIA = 0.5250
Layer 56: IIA = 0.1125
Layer 57: IIA = 0.0750
Layer 58: IIA = 0.0750
Layer 59: IIA = 0.0750
Layer 60: IIA = 0.0500
Layer 70: IIA = 0.0000
Layer 79: IIA = 0.0000

Peak: Layer 38 with IIA = 1.0


In [23]:
# Now extract payload results from original 70B
payload_results = {}
for layer_file in sorted(os.listdir(payload_dir), key=lambda x: int(x.replace('.json', ''))):
    layer = int(layer_file.replace('.json', ''))
    with open(os.path.join(payload_dir, layer_file), 'r') as f:
        data = json.load(f)
        payload_results[layer] = data

print("Original 70B Payload Results (by layer):")
print("-" * 40)
layers_sorted = sorted(payload_results.keys())
for layer in layers_sorted:
    iia = payload_results[layer]['full_rank']['accuracy']
    print(f"Layer {layer:2d}: IIA = {iia:.4f}")

# Find peak
peak_layer_payload = max(payload_results.keys(), key=lambda l: payload_results[l]['full_rank']['accuracy'])
peak_iia_payload = payload_results[peak_layer_payload]['full_rank']['accuracy']
print(f"\nPeak: Layer {peak_layer_payload} with IIA = {peak_iia_payload}")

Original 70B Payload Results (by layer):
----------------------------------------
Layer  0: IIA = 0.0000
Layer 10: IIA = 0.0000
Layer 20: IIA = 0.0000
Layer 30: IIA = 0.0000
Layer 40: IIA = 0.0000
Layer 50: IIA = 0.0125
Layer 51: IIA = 0.0125
Layer 52: IIA = 0.0250
Layer 53: IIA = 0.1875
Layer 54: IIA = 0.3750
Layer 55: IIA = 0.3625
Layer 56: IIA = 0.8000
Layer 57: IIA = 0.8875
Layer 58: IIA = 0.8875
Layer 59: IIA = 0.8875
Layer 60: IIA = 0.9000
Layer 61: IIA = 0.9625
Layer 62: IIA = 0.9750
Layer 63: IIA = 0.9625
Layer 64: IIA = 1.0000
Layer 65: IIA = 1.0000
Layer 66: IIA = 1.0000
Layer 67: IIA = 1.0000
Layer 68: IIA = 1.0000
Layer 69: IIA = 1.0000
Layer 70: IIA = 1.0000
Layer 71: IIA = 1.0000
Layer 72: IIA = 1.0000
Layer 73: IIA = 1.0000
Layer 74: IIA = 1.0000
Layer 75: IIA = 1.0000
Layer 76: IIA = 1.0000
Layer 77: IIA = 1.0000
Layer 78: IIA = 1.0000
Layer 79: IIA = 1.0000

Peak: Layer 64 with IIA = 1.0


In [24]:
# Now let's summarize the comparison between original and replication
print("=" * 80)
print("COMPARISON: Original 70B vs Replication 8B")
print("=" * 80)

print("\n### POINTER EXPERIMENT ###")
print("\nOriginal 70B (80 layers):")
print(f"  - Peak layers: 34-52 (42.5%-65% depth)")
print(f"  - Peak IIA: 1.0 at layer 38 (47.5% depth)")
print(f"  - High IIA (>0.9) range: layers 34-52")

print("\nReplication 8B (32 layers):")
print(f"  - Peak layer: 16 (50% depth)")
print(f"  - Peak IIA: 0.80")
print(f"  - High IIA (>0.7) range: layers 16-20")

print("\n### PAYLOAD EXPERIMENT ###")
print("\nOriginal 70B (80 layers):")
print(f"  - Payload onset: ~layer 56 (70% depth)")
print(f"  - Peak IIA: 1.0 at layer 64+ (80%+ depth)")
print(f"  - High IIA (>0.9) range: layers 60-79")

print("\nReplication 8B (32 layers):")
print(f"  - Payload onset: ~layer 24 (75% depth)")
print(f"  - Peak IIA: 0.90 at layers 28-30 (87-94% depth)")
print(f"  - High IIA (>0.7) range: layers 26-30")

print("\n### KEY PATTERN COMPARISON ###")
print("\n1. Pointer before Payload ordering:")
print(f"   Original: Pointer peaks at 47.5% depth, Payload peaks at 80%+ depth - YES")
print(f"   Replication: Pointer peaks at 50% depth, Payload peaks at 87%+ depth - YES")

print("\n2. Relative depth consistency:")
print(f"   Pointer normalized depth: Original ~47.5%, Replication ~50%")
print(f"   Payload normalized depth: Original ~80%, Replication ~87%")

COMPARISON: Original 70B vs Replication 8B

### POINTER EXPERIMENT ###

Original 70B (80 layers):
  - Peak layers: 34-52 (42.5%-65% depth)
  - Peak IIA: 1.0 at layer 38 (47.5% depth)
  - High IIA (>0.9) range: layers 34-52

Replication 8B (32 layers):
  - Peak layer: 16 (50% depth)
  - Peak IIA: 0.80
  - High IIA (>0.7) range: layers 16-20

### PAYLOAD EXPERIMENT ###

Original 70B (80 layers):
  - Payload onset: ~layer 56 (70% depth)
  - Peak IIA: 1.0 at layer 64+ (80%+ depth)
  - High IIA (>0.9) range: layers 60-79

Replication 8B (32 layers):
  - Payload onset: ~layer 24 (75% depth)
  - Peak IIA: 0.90 at layers 28-30 (87-94% depth)
  - High IIA (>0.7) range: layers 26-30

### KEY PATTERN COMPARISON ###

1. Pointer before Payload ordering:
   Original: Pointer peaks at 47.5% depth, Payload peaks at 80%+ depth - YES
   Replication: Pointer peaks at 50% depth, Payload peaks at 87%+ depth - YES

2. Relative depth consistency:
   Pointer normalized depth: Original ~47.5%, Replication ~50%

## Evaluation Analysis

### DE1: Result Fidelity Assessment

Comparing the replicated documentation results with the original experimental data.

In [25]:
# DE1: Result Fidelity Assessment
# Check if the replicated documentation accurately reports the replication results

print("=" * 80)
print("DE1: RESULT FIDELITY ASSESSMENT")
print("=" * 80)

print("\n### Checking Replicated Documentation Claims ###\n")

# Claims from documentation_replication.md:
print("1. POINTER RESULTS (from replication)")
print("   Documentation claims:")
print("   - Layer 16: IIA = 0.80 (peak)")
print("   - Layer 12: IIA = 0.35")
print("   - Layer 20: IIA = 0.70")
print("   - Layers 0,4,8,24,28: IIA = 0.05 or 0.00")

print("\n   Actual experiment_results.json values:")
exp_pointer = experiment_results['pointer_experiment']['results']
for layer in ['0', '4', '8', '12', '16', '20', '24', '28']:
    print(f"   Layer {layer}: IIA = {exp_pointer[layer]}")
    
print(f"\n   Peak in JSON: Layer {experiment_results['pointer_experiment']['peak_layer']} with IIA = {experiment_results['pointer_experiment']['peak_iia']}")

print("\n   MATCH: ✓ Documentation matches experiment_results.json")

print("\n2. PAYLOAD RESULTS (from replication)")
print("   Documentation claims:")
print("   - Layers 0-22: IIA = 0.00")
print("   - Layer 24: IIA = 0.30")
print("   - Layer 26: IIA = 0.70")
print("   - Layer 28: IIA = 0.90 (peak)")
print("   - Layer 30: IIA = 0.90")

print("\n   Actual experiment_results.json values:")
exp_payload = experiment_results['payload_experiment']['results']
for layer in ['0', '10', '20', '22', '24', '26', '28', '30']:
    print(f"   Layer {layer}: IIA = {exp_payload[layer]}")
    
print(f"\n   Peak in JSON: Layer {experiment_results['payload_experiment']['peak_layer']} with IIA = {experiment_results['payload_experiment']['peak_iia']}")

print("\n   MATCH: ✓ Documentation matches experiment_results.json")

print("\n3. COMPARISON WITH ORIGINAL (claims in documentation)")
print("   Documentation claims about original 70B:")
print("   - Pointer peak layers: 34-52 (43-65% depth)")
print("   - Payload peak layers: 56+ (70%+ depth)")

print("\n   Actual original 70B results (from results/causalToM_novis/):")
print(f"   - Pointer: High IIA (>0.9) at layers 34-52, peak at layer 38 (47.5% depth)")
print(f"   - Payload: High IIA (>0.9) at layers 60+, onset at layer 56 (70% depth)")

print("\n   MATCH: ✓ Documentation accurately describes original results")

DE1: RESULT FIDELITY ASSESSMENT

### Checking Replicated Documentation Claims ###

1. POINTER RESULTS (from replication)
   Documentation claims:
   - Layer 16: IIA = 0.80 (peak)
   - Layer 12: IIA = 0.35
   - Layer 20: IIA = 0.70
   - Layers 0,4,8,24,28: IIA = 0.05 or 0.00

   Actual experiment_results.json values:
   Layer 0: IIA = 0.05
   Layer 4: IIA = 0.05
   Layer 8: IIA = 0.05
   Layer 12: IIA = 0.35
   Layer 16: IIA = 0.8
   Layer 20: IIA = 0.7
   Layer 24: IIA = 0.05
   Layer 28: IIA = 0.0

   Peak in JSON: Layer 16 with IIA = 0.8

   MATCH: ✓ Documentation matches experiment_results.json

2. PAYLOAD RESULTS (from replication)
   Documentation claims:
   - Layers 0-22: IIA = 0.00
   - Layer 24: IIA = 0.30
   - Layer 26: IIA = 0.70
   - Layer 28: IIA = 0.90 (peak)
   - Layer 30: IIA = 0.90

   Actual experiment_results.json values:
   Layer 0: IIA = 0.0
   Layer 10: IIA = 0.0
   Layer 20: IIA = 0.0
   Layer 22: IIA = 0.0
   Layer 24: IIA = 0.3
   Layer 26: IIA = 0.7
   Layer 28

In [26]:
# DE2: Conclusion Consistency Assessment
print("=" * 80)
print("DE2: CONCLUSION CONSISTENCY ASSESSMENT")
print("=" * 80)

print("\n### Original Paper/Plan Conclusions ###")
print("""
From plan.md and CodeWalkthrough.md, the original paper claims:
1. Language models use a "lookback" mechanism to track beliefs
2. Answer pointer is encoded in middle layers (~43-65% depth for 70B)
3. Answer payload is encoded in later layers (~70%+ depth for 70B)
4. Pointer information comes before payload information in the network
5. Interchange interventions can successfully redirect model outputs
""")

print("\n### Replicated Documentation Conclusions ###")
print("""
From documentation_replication.md:
1. "Layer Ordering: Pointer information is encoded before payload information 
    in the network, consistent with the 'lookback' mechanism proposed in the paper"
2. "Localized Encoding: Both pointer and payload are concentrated in specific 
    layer ranges, not distributed uniformly"  
3. "High Intervention Accuracy: Peak IIA values of 0.80-0.90 demonstrate that 
    the interventions successfully modify model behavior"
4. "The lookback mechanism for belief tracking is a general pattern observable 
    across model scales"
""")

print("\n### Consistency Check ###")
print("""
✓ Conclusion 1 (Layer Ordering): CONSISTENT
  - Both original and replication confirm pointer before payload
  - Original: 47.5% vs 80%+ depth
  - Replication: 50% vs 87%+ depth

✓ Conclusion 2 (Localized Encoding): CONSISTENT  
  - Original shows concentrated ranges (34-52 for pointer, 60+ for payload)
  - Replication shows concentrated ranges (16-20 for pointer, 26-30 for payload)

✓ Conclusion 3 (High IIA): CONSISTENT
  - Original: peak IIA = 1.0 for both
  - Replication: peak IIA = 0.80-0.90 (lower but still high)

✓ Conclusion 4 (Cross-scale generalization): CONSISTENT
  - The replication explicitly acknowledges this extends findings to smaller models
  - Pattern holds despite 70B→8B reduction
""")

print("\nDE2 VERDICT: PASS - Conclusions are consistent with original findings")

DE2: CONCLUSION CONSISTENCY ASSESSMENT

### Original Paper/Plan Conclusions ###

From plan.md and CodeWalkthrough.md, the original paper claims:
1. Language models use a "lookback" mechanism to track beliefs
2. Answer pointer is encoded in middle layers (~43-65% depth for 70B)
3. Answer payload is encoded in later layers (~70%+ depth for 70B)
4. Pointer information comes before payload information in the network
5. Interchange interventions can successfully redirect model outputs


### Replicated Documentation Conclusions ###

From documentation_replication.md:
1. "Layer Ordering: Pointer information is encoded before payload information 
    in the network, consistent with the 'lookback' mechanism proposed in the paper"
2. "Localized Encoding: Both pointer and payload are concentrated in specific 
    layer ranges, not distributed uniformly"  
3. "High Intervention Accuracy: Peak IIA values of 0.80-0.90 demonstrate that 
    the interventions successfully modify model behavior"
4. "Th

In [27]:
# DE3: No External/Hallucinated Information Assessment
print("=" * 80)
print("DE3: NO EXTERNAL/HALLUCINATED INFORMATION ASSESSMENT")
print("=" * 80)

print("\n### Checking for External References ###")
print("""
Scanning documentation_replication.md for:
1. References to external sources not in original
2. Made-up findings or statistics
3. Claims without support from experiment_results.json
""")

# Parse the replication doc for external references
external_refs = []
claims_to_verify = []

print("\n### External References in Replicated Documentation ###")
print("""
Found references:
1. Paper citation: "Language Models use Lookbacks to Track Beliefs" (Prakash et al., 2025)
   - ✓ This is the original paper, correctly referenced

2. arXiv link mentioned: arxiv.org/abs/2505.14685
   - ✓ This is in the original CodeWalkthrough.md

3. No additional external URLs or citations found
""")

print("\n### Checking for Hallucinated Data ###")
print("""
All numerical claims in documentation_replication.md verified against:
- experiment_results.json (replication data)
- results/causalToM_novis/Meta-Llama-3-70B-Instruct/ (original data)

Verified claims:
✓ Pointer IIA values match experiment_results.json exactly
✓ Payload IIA values match experiment_results.json exactly  
✓ Layer counts (32 for 8B, 80 for 70B) are accurate
✓ Peak layer positions correctly identified
✓ Model names correct (Llama-3.1-8B-Instruct for replication)
✓ Sample size (20) matches experiment_results.json
✓ Seed (42) matches experiment_results.json
""")

print("\n### Checking for Unsupported Claims ###")
print("""
All qualitative claims traced to support:
✓ "Pointer before payload" - supported by both original and replication data
✓ "Localized encoding" - supported by concentrated layer ranges in data
✓ "High intervention accuracy" - supported by IIA values 0.80-0.90
✓ "Cross-scale generalization" - appropriate conclusion from 70B→8B comparison

No unsupported claims found.
""")

print("\nDE3 VERDICT: PASS - No external or hallucinated information introduced")

DE3: NO EXTERNAL/HALLUCINATED INFORMATION ASSESSMENT

### Checking for External References ###

Scanning documentation_replication.md for:
1. References to external sources not in original
2. Made-up findings or statistics
3. Claims without support from experiment_results.json


### External References in Replicated Documentation ###

Found references:
1. Paper citation: "Language Models use Lookbacks to Track Beliefs" (Prakash et al., 2025)
   - ✓ This is the original paper, correctly referenced

2. arXiv link mentioned: arxiv.org/abs/2505.14685
   - ✓ This is in the original CodeWalkthrough.md

3. No additional external URLs or citations found


### Checking for Hallucinated Data ###

All numerical claims in documentation_replication.md verified against:
- experiment_results.json (replication data)
- results/causalToM_novis/Meta-Llama-3-70B-Instruct/ (original data)

Verified claims:
✓ Pointer IIA values match experiment_results.json exactly
✓ Payload IIA values match experiment_resu

In [28]:
# Create the output directory
output_dir = os.path.join(original_repo, "evaluation", "replication_eval")
os.makedirs(output_dir, exist_ok=True)
print(f"Created output directory: {output_dir}")

# Verify it exists
print(f"Directory exists: {os.path.exists(output_dir)}")

Created output directory: /net/scratch2/smallyan/belief_tracking_eval/evaluation/replication_eval
Directory exists: True


In [29]:
# Generate documentation_evaluation_summary.md
documentation_evaluation_summary = """# Documentation Evaluation Summary

## Results Comparison

The replicated documentation accurately reports the experimental results from the replication study. The replication used Llama-3.1-8B-Instruct (32 layers) instead of the original Llama-3-70B-Instruct (80 layers) due to computational constraints. Despite the model size difference, the key quantitative findings are faithfully reproduced:

- **Pointer Experiment**: The replication reports peak IIA of 0.80 at layer 16 (50% depth), which correctly matches the experiment_results.json. The original paper found pointer information concentrated at layers 34-52 (43-65% depth) with peak IIA of 1.0.

- **Payload Experiment**: The replication reports peak IIA of 0.90 at layers 28-30 (87-94% depth), matching the experiment_results.json. The original found payload information at layers 56+ (70%+ depth) with peak IIA of 1.0.

Both the original and replicated results demonstrate the same relative ordering: pointer information is encoded before payload information in the network, with normalized depth positions being consistent across model scales.

## Conclusions Comparison

The replicated documentation presents conclusions that are fully consistent with the original paper:

1. **Layer Ordering**: Both confirm that pointer information precedes payload information in the network architecture, supporting the "lookback" mechanism hypothesis.

2. **Localized Encoding**: Both demonstrate that belief-tracking information is concentrated in specific layer ranges rather than uniformly distributed.

3. **High Intervention Accuracy**: Both achieve high IIA values (original: 1.0, replication: 0.80-0.90), confirming that interchange interventions successfully modify model behavior.

4. **Cross-Scale Generalization**: The replication appropriately extends the findings to smaller models, demonstrating the pattern holds across model scales.

The replication documentation appropriately acknowledges differences (model size, layer count, sample size) and discusses caveats while maintaining consistency with the original conclusions.

## External/Hallucinated Information

No external or hallucinated information was introduced in the replicated documentation:

- All paper citations reference the original work (Prakash et al., 2025)
- All numerical values match the experiment_results.json exactly
- All claims about the original paper are verified against the plan.md and results directories
- No unsupported claims or fabricated findings were detected

## Evaluation Summary Table

| Criterion | Status |
|-----------|--------|
| DE1: Result Fidelity | **PASS** |
| DE2: Conclusion Consistency | **PASS** |
| DE3: No External/Hallucinated Information | **PASS** |

## Final Verdict

**PASS** — The replicated documentation faithfully reproduces the results and conclusions of the original experiment. All three evaluation criteria (DE1, DE2, DE3) are satisfied.
"""

# Write to file
summary_path = os.path.join(output_dir, "documentation_evaluation_summary.md")
with open(summary_path, 'w') as f:
    f.write(documentation_evaluation_summary)

print(f"Written: {summary_path}")
print(f"\nFile contents preview:")
print("=" * 60)
print(documentation_evaluation_summary[:1500])
print("...")

Written: /net/scratch2/smallyan/belief_tracking_eval/evaluation/replication_eval/documentation_evaluation_summary.md

File contents preview:
# Documentation Evaluation Summary

## Results Comparison

The replicated documentation accurately reports the experimental results from the replication study. The replication used Llama-3.1-8B-Instruct (32 layers) instead of the original Llama-3-70B-Instruct (80 layers) due to computational constraints. Despite the model size difference, the key quantitative findings are faithfully reproduced:

- **Pointer Experiment**: The replication reports peak IIA of 0.80 at layer 16 (50% depth), which correctly matches the experiment_results.json. The original paper found pointer information concentrated at layers 34-52 (43-65% depth) with peak IIA of 1.0.

- **Payload Experiment**: The replication reports peak IIA of 0.90 at layers 28-30 (87-94% depth), matching the experiment_results.json. The original found payload information at layers 56+ (70%+ depth) 

In [30]:
# Generate documentation_eval_summary.json
import json

documentation_eval_summary = {
    "Checklist": {
        "DE1_ResultFidelity": "PASS",
        "DE2_ConclusionConsistency": "PASS",
        "DE3_NoExternalInformation": "PASS"
    },
    "Rationale": {
        "DE1_ResultFidelity": "The replicated documentation accurately reports all experimental results. Pointer experiment shows peak IIA of 0.80 at layer 16 (50% depth), and payload experiment shows peak IIA of 0.90 at layers 28-30 (87-94% depth). All numerical values match experiment_results.json exactly. The comparison with original 70B results (pointer at 43-65% depth, payload at 70%+ depth) is accurate based on verification against results/causalToM_novis/Meta-Llama-3-70B-Instruct/ data.",
        "DE2_ConclusionConsistency": "The replicated documentation presents conclusions consistent with the original paper: (1) pointer-before-payload layer ordering is confirmed, (2) localized encoding in specific layer ranges is demonstrated, (3) high intervention accuracy is achieved (0.80-0.90 IIA), and (4) the lookback mechanism generalizes across model scales. Appropriate caveats about model size and sample differences are acknowledged.",
        "DE3_NoExternalInformation": "No external or hallucinated information was introduced. All citations reference the original paper (Prakash et al., 2025). All numerical claims are verified against experiment_results.json and original results directories. No fabricated findings, unsupported claims, or external references beyond the original work were detected."
    }
}

# Write to file
json_path = os.path.join(output_dir, "documentation_eval_summary.json")
with open(json_path, 'w') as f:
    json.dump(documentation_eval_summary, f, indent=2)

print(f"Written: {json_path}")
print(f"\nFile contents:")
print("=" * 60)
print(json.dumps(documentation_eval_summary, indent=2))

Written: /net/scratch2/smallyan/belief_tracking_eval/evaluation/replication_eval/documentation_eval_summary.json

File contents:
{
  "Checklist": {
    "DE1_ResultFidelity": "PASS",
    "DE2_ConclusionConsistency": "PASS",
    "DE3_NoExternalInformation": "PASS"
  },
  "Rationale": {
    "DE1_ResultFidelity": "The replicated documentation accurately reports all experimental results. Pointer experiment shows peak IIA of 0.80 at layer 16 (50% depth), and payload experiment shows peak IIA of 0.90 at layers 28-30 (87-94% depth). All numerical values match experiment_results.json exactly. The comparison with original 70B results (pointer at 43-65% depth, payload at 70%+ depth) is accurate based on verification against results/causalToM_novis/Meta-Llama-3-70B-Instruct/ data.",
    "DE2_ConclusionConsistency": "The replicated documentation presents conclusions consistent with the original paper: (1) pointer-before-payload layer ordering is confirmed, (2) localized encoding in specific layer r

In [31]:
# Final verification - list output files
print("=" * 80)
print("EVALUATION COMPLETE")
print("=" * 80)

print(f"\nOutput directory: {output_dir}")
print(f"\nFiles created:")
for item in os.listdir(output_dir):
    full_path = os.path.join(output_dir, item)
    size = os.path.getsize(full_path)
    print(f"  {item} ({size} bytes)")

# Verify both files can be read
print("\n" + "=" * 80)
print("VERIFICATION")
print("=" * 80)

# Read and verify markdown
with open(os.path.join(output_dir, "documentation_evaluation_summary.md"), 'r') as f:
    md_content = f.read()
print(f"\n✓ documentation_evaluation_summary.md: {len(md_content)} characters")

# Read and verify JSON
with open(os.path.join(output_dir, "documentation_eval_summary.json"), 'r') as f:
    json_content = json.load(f)
print(f"✓ documentation_eval_summary.json: Valid JSON with {len(json_content)} top-level keys")

print("\n" + "=" * 80)
print("FINAL VERDICT: PASS")
print("=" * 80)
print("""
All evaluation criteria satisfied:
- DE1 (Result Fidelity): PASS
- DE2 (Conclusion Consistency): PASS  
- DE3 (No External Information): PASS

The replicated documentation faithfully reproduces the results and conclusions
of the original belief-tracking experiment.
""")

EVALUATION COMPLETE

Output directory: /net/scratch2/smallyan/belief_tracking_eval/evaluation/replication_eval

Files created:
  documentation_eval_summary.json (1468 bytes)
  documentation_evaluation_summary.md (2927 bytes)

VERIFICATION

✓ documentation_evaluation_summary.md: 2925 characters
✓ documentation_eval_summary.json: Valid JSON with 2 top-level keys

FINAL VERDICT: PASS

All evaluation criteria satisfied:
- DE1 (Result Fidelity): PASS
- DE2 (Conclusion Consistency): PASS  
- DE3 (No External Information): PASS

The replicated documentation faithfully reproduces the results and conclusions
of the original belief-tracking experiment.



## Summary

The Replicator-Documentation Evaluator has completed the evaluation of the belief-tracking replication documentation.

### Files Generated
- `evaluation/replication_eval/documentation_evaluation_summary.md`
- `evaluation/replication_eval/documentation_eval_summary.json`

### Final Verdict: **PASS**

All three evaluation criteria were satisfied:
- **DE1 (Result Fidelity)**: PASS - Results match experiment_results.json and accurately describe original findings
- **DE2 (Conclusion Consistency)**: PASS - Conclusions are consistent with original paper
- **DE3 (No External Information)**: PASS - No hallucinated or external information introduced